# Conversion Prediction Model (XGBoost)

This notebook trains an XGBoost classifier to predict `purchase_completed` based on ecommerce user session metrics.

**Inputs:** `conversion_ready.csv` (processed Phase 4A data)
**Output:** `backend/models/conversion.pkl`

In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import xgboost as xgb

sns.set_theme(style="darkgrid")

## 1. Load Data & Explore

In [ ]:
data_path = os.path.join("..", "processed", "conversion_ready.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
display(df.head())

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='purchase_completed', palette='viridis')
plt.title('Target Variable Distribution (purchase_completed)')
plt.show()

## 2. Preprocessing & Encoding

In [ ]:
y = df['purchase_completed'].astype(int)
X = df.drop(columns=['purchase_completed'])

le_category = LabelEncoder()
X['product_category'] = le_category.fit_transform(X['product_category'].astype(str))

le_time = LabelEncoder()
X['time_of_day'] = le_time.fit_transform(X['time_of_day'].astype(str))

X['repeat_user'] = X['repeat_user'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set size: {len(X_train)} | Test set size: {len(X_test)}")

## 3. Train XGBoost Classifier

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    use_label_encoder=False, 
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

## 4. Evaluation Metrics

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC AUC Score: {roc_auc:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 5. Feature Importance

In [ ]:
feat_imps = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x='Importance', y='Feature', data=feat_imps, palette='magma')
plt.title('XGBoost Feature Importance')
plt.show()

## 6. Export Model Bundle

In [ ]:
models_dir = os.path.join("..", "..", "backend", "models")
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, "conversion.pkl")

bundle = {
    'model': model,
    'le_category': le_category,
    'le_time': le_time,
    'features': list(X.columns)
}
joblib.dump(bundle, model_path)
print(f"Saved Model successfully to {model_path}")